In [1]:
import pandas as pd
from pathlib import Path

In [ ]:
ECOLOGY_DIR = Path("data/ecology")

csv_files = sorted(ECOLOGY_DIR.glob("*.csv"))

cleaned_dataframes = {}

for file_path in csv_files:
    print(f"--- Traitement de {file_path.name} ---")

    df = pd.read_csv(file_path)
    print(f"shape: {df.shape}\n")

    # supression des doublons
    nb_doublons = df.duplicated().sum()
    if nb_doublons > 0:
        print(f"{nb_doublons} ligne dupliquée trouvée")
        df = df.drop_duplicates()

    #supression colonnes inutiles
    for col in df.columns:
        if df[col].nunique() == 1:
            print(f"Colonne '{col}' supprimée car elle contient une seule valeur unique")
            df = df.drop(columns=[col])
        if df[col].isnull().sum() / len(df) > 0.5:
            print(f"Colonne '{col}' supprimée car elle contient plus de 50% de valeurs nulles")
            df = df.drop(columns=[col])

    #supression des valeurs nulles
    nb_nulls = df.isnull().sum().sum()
    if nb_nulls > 0:
        print(f"{nb_nulls} valeurs nulles trouvées")
        df = df.dropna()

    #supression des valeurs abérrantes basé sur l'écart interquartile
    numeric_cols = df.select_dtypes(include=['number']).columns
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        if not outliers.empty:
            print(f"{len(outliers)} valeurs abérrantes trouvées dans la colonne '{col}'")
            df = df[~((df[col] < lower_bound) | (df[col] > upper_bound))]




    # TODO: étapes de nettoyage à ajouter ici

    cleaned_dataframes[file_path.stem] = df
    print(f"shape: {df.shape}\n")

print(cleaned_dataframes.keys())


--- Traitement de bat_surveys.csv ---
shape: (4994, 14)

2 ligne dupliquée trouvée
20 valeurs nulles trouvées
154 valeurs abérrantes trouvées dans la colonne 'colony_estimate'
29 valeurs abérrantes trouvées dans la colonne 'comparison_species_count'
shape: (4789, 14)

--- Traitement de environmental_field_reports.csv ---
shape: (30, 12)

1 valeurs nulles trouvées
shape: (29, 12)

--- Traitement de environmental_monthly.csv ---
shape: (4994, 11)

2 ligne dupliquée trouvée
4813 valeurs nulles trouvées
2 valeurs abérrantes trouvées dans la colonne 'data_completeness_pct'
shape: (199, 11)

--- Traitement de monitoring_device_operations.csv ---
shape: (4994, 11)

2 ligne dupliquée trouvée
4833 valeurs nulles trouvées
1 valeurs abérrantes trouvées dans la colonne 'acoustic_quality'
Colonne 'record_status' supprimée car elle contient une seule valeur unique


KeyError: 'record_status'

In [12]:
df = pd.read_csv(ECOLOGY_DIR / "bat_surveys.csv")
print("min:", df["colony_estimate"].min())
print("max:", df["colony_estimate"].max())
print("mean:", df["colony_estimate"].mean())

q1 = df["colony_estimate"].quantile(0.25)
q3 = df["colony_estimate"].quantile(0.75)
print("Q1:", q1)
print("Q3:", q3)
lower_bound = q1 - 1.5 * (q3 - q1)
upper_bound = q3 + 1.5 * (q3 - q1)
print("lower bound:", lower_bound)
print("upper bound:", upper_bound)


min: 0.0
max: 99.6
mean: 15.826511814177014
Q1: 9.3
Q3: 20.5
lower bound: -7.4999999999999964
upper bound: 37.3
